# 韩国关税署2018–2022前20日原稿：官方门户逐月索引

固定60个日期窗口，保存每次官方列表原始响应；来源身份检索，无半导体因子、SOXX结果、拟合或账户。


In [ ]:
import hashlib,json,os,pathlib,tarfile,tempfile,subprocess,shlex,time,urllib.request,urllib.parse,urllib.error,datetime,re
import pandas as pd,qlib
from bs4 import BeautifulSoup
from qlib.workflow import R
from src.data.nas_market_data import nas_api_call,archive_nas_recorder,_settings,_ssh
root=pathlib.Path.cwd();card=root/'research/kcs_early20_2018_2022_portal_index_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(pathlib.Path(p).read_bytes()).hexdigest();assert plan['frozen'] and not (card/'run_state.json').exists()
snapshot=root/'data/snapshots/kcs_early20_2018_2022_portal_index_20261023';snapshot.mkdir(parents=True,exist_ok=True);months=[(y,m) for y in range(2018,2023) for m in range(1,13)];assert len(months)==plan['budget']['months']==60
state={'status':'GETs_started','plan_sha256':sha(card/'plan.json'),'GET_attempts':0,'GET_completed':0,'model_fits':0,'accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
receipts=[];rows=[]
for year,month in months:
 key=f'{year}-{month:02d}';params={'startDate':f'{key}-20','endDate':f'{key}-27','srchWord':'수출입','repCode':'B00003','pageIndex':1};url=plan['query_contract']['base_url']+'?'+urllib.parse.urlencode(params);started=datetime.datetime.now(datetime.timezone.utc).isoformat();body=b'';headers={};status=None;error=None;final_url=url
 state['GET_attempts']+=1;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 try:
  request=urllib.request.Request(url,headers={'User-Agent':plan['query_contract']['user_agent']});response=urllib.request.urlopen(request,timeout=plan['query_contract']['timeout_seconds_each']);status=response.status;final_url=response.url;headers=dict(response.headers.items());body=response.read(plan['query_contract']['max_response_bytes_each']+1);response.close();state['GET_completed']+=1
 except urllib.error.HTTPError as exc:
  status=exc.code;final_url=exc.url;headers=dict(exc.headers.items());body=exc.read(plan['query_contract']['max_response_bytes_each']+1);error='HTTPError'
 except Exception as exc:error=type(exc).__name__+': '+str(exc)[:180]
 ended=datetime.datetime.now(datetime.timezone.utc).isoformat();(snapshot/(key+'.response')).write_bytes(body);(snapshot/(key+'.headers.json')).write_text(json.dumps(headers,ensure_ascii=False,indent=2)+'\n')
 content_type=headers.get('Content-Type',headers.get('content-type',''));candidates=[];accepted=[];parse_status='unknown'
 if status==200 and len(body)<=plan['query_contract']['max_response_bytes_each'] and 'html' in content_type.lower():
  soup=BeautifulSoup(body,'html.parser')
  for a in soup.find_all('a'):
   href=a.get('href') or ''
   if 'pressReleaseView.do' not in href:continue
   title=a.get_text(' ',strip=True);compact=re.sub(r'\s+','',title);id=(urllib.parse.parse_qs(urllib.parse.urlsplit(href).query).get('newsId') or [None])[0]
   if id is None:continue
   info={'newsId':id,'title_excerpt':title[:240],'href':urllib.parse.urljoin(url,href)};candidates.append(info)
   if (f'{month}월1일' in compact and '20일' in compact and '수출입현황' in compact and '관세청' in compact):accepted.append(info)
  parse_status='one_official_list_candidate' if len(accepted)==1 else ('ambiguous' if len(accepted)>1 else 'missing')
 else:parse_status='transport_or_format_unknown'
 receipt={'month':key,'url':url,'final_url':final_url,'started_utc':started,'ended_utc':ended,'http_status':status,'content_type':content_type,'body_sha256':hashlib.sha256(body).hexdigest(),'body_bytes':len(body),'headers_sha256':sha(snapshot/(key+'.headers.json')),'error':error,'parse_status':parse_status,'candidate_count':len(candidates),'accepted_count':len(accepted),'candidates':candidates,'accepted':accepted}
 receipts.append(receipt);rows.append({'month':key,'http_status':status,'parse_status':parse_status,'candidate_count':len(candidates),'accepted_count':len(accepted),'newsId':accepted[0]['newsId'] if len(accepted)==1 else None,'source_url':accepted[0]['href'] if len(accepted)==1 else None});(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print('indexed',key,parse_status,flush=True);time.sleep(plan['query_contract']['rate_pause_seconds'])
assert state['GET_attempts']==60;state['status']='GETs_finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
manifest={'plan_sha256':sha(card/'plan.json'),'requests':receipts};(snapshot/'request_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+'\n');files=sorted(p for p in snapshot.iterdir() if p.is_file());assert len(files)==121;settings=_settings(root)
with tempfile.TemporaryDirectory(prefix='kcs-early20-index-') as temp:
 archive=pathlib.Path(temp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as t:
  for p in files:t.add(p,arcname=str(p.relative_to(root)),recursive=False)
 digest=sha(archive);receipt=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact'];assert receipt['sha256']==digest and receipt['size_bytes']==archive.stat().st_size
 remote_path=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest;assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote_path)).split()[0]==digest
 source_archive={**receipt,'nas_path':remote_path,'request_count':60,'files':len(files),'request_manifest_sha256':sha(snapshot/'request_manifest.json')}
 table=pd.DataFrame(rows);counts=table.parse_status.value_counts().to_dict();result={'status':'official_portal_2018_2022_early20_page_index_partial_or_complete_source_only','plan_sha256':sha(card/'plan.json'),'months':60,'GET_attempts':state['GET_attempts'],'GET_completed':state['GET_completed'],'parse_status_counts':counts,'one_candidate_months':table.loc[table.accepted_count.eq(1),'month'].tolist(),'unknown_months':table.loc[table.accepted_count.ne(1),'month'].tolist(),'source_archive':source_archive,'original_request_manifest_sha256':sha(snapshot/'request_manifest.json'),'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0};(card/'result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
with tempfile.TemporaryDirectory(prefix='kcs-early20-index-cold-') as temp:
 copy=pathlib.Path(temp)/'source.tar.gz'
 with copy.open('wb') as out:subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote_path)],stdout=out,stderr=subprocess.PIPE,check=True)
 assert copy.stat().st_size==source_archive['size_bytes'] and sha(copy)==digest
 with tarfile.open(copy,'r:gz') as t:
  for name in [str((snapshot/'2018-01.response').relative_to(root)),str((snapshot/'2019-10.response').relative_to(root)),str((snapshot/'2022-12.response').relative_to(root)),str((snapshot/'request_manifest.json').relative_to(root))]:assert t.extractfile(name).read()==(root/name).read_bytes()
source_readback={'status':'NAS_source_archive_hash_and_four_selected_members_cold_read_exact','archive_sha256':digest,'archive_bytes':source_archive['size_bytes'],'selected_members':4};(card/'source_restore.json').write_text(json.dumps(source_readback,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/'data/qlib/minute_information_pit_allocator_20261012/daily_prices'),region='us',kernels=1)
with R.start(experiment_name=plan['experiment'],uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_archive=source_archive);rec.save_objects(artifact_path='analysis',monthly_index=table,query_receipts=receipts,result=result);rec.log_metrics(months=60,one_candidate_months=int(table.accepted_count.eq(1).sum()),GET_attempts=60,model_fits=0,native_accounts=0);rid,eid=rec.id,str(rec.experiment_id)
state['status']='finished';state['recorder_id']=rid;state['experiment_id']=eid;(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
rec_archive=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rid,experiment_id=eid);assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(rec_archive['nas_path'])).split()[0]==rec_archive['sha256']
summary={'experiment':plan['experiment'],'decision':'official_portal_early20_2018_2022_index_source_only_not_feature_qualified','objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_archive':source_archive,'source_restore':'source_restore.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(card/'qlib_recorder'),'NAS_evidence':rec_archive,'NAS_readback':{'sha256':rec_archive['sha256'],'verified':True},'SOXX_outcome_reads':0,'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/monthly_index','analysis/query_receipts','analysis/result']},'limitations':['A list-page match is not the original press-release attachment or first-public byte proof.','No semiconductor numeric rows, Qlib factor, SOXX outcome or strategy account were produced.','Unknown or ambiguous months stay missing for a later separately frozen source repair.']};(card/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2)+'\n')
print(json.dumps({'decision':summary['decision'],'counts':counts,'unknown':result['unknown_months'],'source_sha256':digest,'recorder_id':rid,'Recorder_NAS_sha256':rec_archive['sha256']},ensure_ascii=False,indent=2))


## 中文结论

按冻结的海关机构筛选和每月20–27日窗口，**60个月各执行一次官方门户列表GET，零重试**；每份原始响应、头、UTC接收时钟及SHA已归档 Yanxuan NAS，源归档 SHA-256 `1efba6b1546eb987600f93fa94ba2bc0e6ad30f98bf19e11a30c690bf3814a51`、Recorder归档 SHA-256 `ad1b1ce2b0843ecb1dacbbd3c5243b3bc2cfe81fb6d28ade5c2fb1af9d67ace1`，远端哈希与限定冷读通过。官方列表固定规则得到 **56个唯一候选、3个无匹配、1个月多候选**；四个未知月为 **2019-01、05、07、08**。原生Qlib Recorder `0cd12e0dfc0c4c1fa1329e71c2040525` 保存60行逐月索引及全部查询回执。

未知是**本版标题匹配状态**，不能解释为海关没有发布：已保存的列表原文里，2019-05/07/08 分别出现“5.1-20”“7.1~20日”“8.1-20”等简写标题；2019-01有两个近似“1日∼20日”页面 ID `156313866` 与 `156313813`。本版本不改检索词或自动挑其中一页；下版只可对这四个月冻结机械标题识别与1月两页原文/附件身份核对，不能重跑其余56个月。

决定 `official_portal_early20_2018_2022_index_source_only_not_feature_qualified`。这是**发布页线索**，尚未核验60份原始附件、半导体出口数值、历史首次在线字节、修订和美股决策前可用时点，不能构建连续 PIT 因子。0 SOXX结果读取、0模型拟合、0完整Stage1账户或合格前向日；费用后稳定超过SOXX的目标仍未实现。
